In [2]:
# Read in the dataset

import pandas as pd
import numpy as np

df = pd.read_csv("modelling_dataset.csv")
df

,Zone_Int_ID,Time_Stamp,Mean_Severity,Mean_Distance(mi),Mean_Temp(F),Mean_Humidity(%),Mean_Pressure(in),Mean_Visibility(mi),Mean_Wind(mph),Mean_Precip(in),...,City_Miami,Weather_Clear,Weather_Cloudy,Weather_Dust/Windy,Weather_Rain/Drizzle,Weather_Snow/Ice,Weather_Stormy,Weather_Visibility,Day_Night,Accident_Count
0,0,2016-06-14 20:00:00,2.0,0.000,86.0,66.0,29.84,8.0,9.2,0.0,...,False,True,False,False,False,False,False,False,1,1
1,0,2016-06-21 18:00:00,2.0,0.000,82.4,79.0,30.02,10.0,9.2,0.0,...,False,True,False,False,False,False,False,False,1,1
2,0,2016-06-21 20:00:00,2.0,0.000,84.2,66.0,30.01,10.0,8.1,0.0,...,False,True,False,False,False,False,False,False,1,1
3,0,2016-06-22 12:00:00,2.0,0.033,93.2,49.0,30.03,10.0,5.8,0.0,...,False,False,True,False,False,False,False,False,1,1
4,0,2016-06-22 14:00:00,2.0,0.000,94.1,46.5,30.00,10.0,8.1,0.0,...,False,False,True,False,False,False,False,False,1,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
351945,829,2022-09-16 18:00:00,2.0,0.506,81.0,85.0,30.03,10.0,0.0,0.0,...,True,False,True,False,False,False,False,False,0,1
351946,830,2022-09-21 06:00:00,2.0,0.125,78.0,96.0,29.85,10.0,0.0,0.0,...,True,True,False,False,False,False,False,False,1,1
351947,831,2022-12-07 16:00:00,2.0,0.063,72.0,84.0,30.19,10.0,6.0,0.0,...,True,True,False,False,False,False,False,False,0,1
351948,832,2022-12-22 16:00:00,2.0,2.241,83.0,58.0,29.93,10.0,8.0,0.0,...,True,False,True,False,False,False,False,False,1,1


In [4]:
y = df['Accident_Count']

mean = y.mean()
variance = y.var()

print(f"Mean:     {mean:.2f}")
print(f"Variance: {variance:.2f}")
print(f"Ratio:    {variance/mean:.2f}")

# Distribution of counts
print(f"\nCount distribution:")
print(y.value_counts().sort_index().head(10))
print(f"\nZeros: {(y == 0).sum()} ({(y == 0).mean()*100:.1f}%)")
print(f"Max count in a single bucket: {y.max()}")

Mean:     1.43
Variance: 3.04
Ratio:    2.13

Count distribution:
Accident_Count
1     284295
2      44443
3      10563
4       4273
5       2172
6       1323
7        929
8        664
9        531
10       410
Name: count, dtype: int64

Zeros: 0 (0.0%)
Max count in a single bucket: 69


In [5]:
d2 = df.copy()
d2['Time_Stamp'] = pd.to_datetime(d2['Time_Stamp'])

# The zone exposure is how much oppertunity there was for accidents to occur since some zones are observed more than others
zone_exposure = d2.groupby('Zone_Int_ID')['Time_Stamp'].apply(
    lambda x: x.dt.date.nunique()
).reset_index(name='Observed_Days')

# Each day has 12 two-hour slots
zone_exposure['Total_2hr_Slots'] = zone_exposure['Observed_Days'] * 12
zone_exposure['log_exposure'] = np.log(zone_exposure['Total_2hr_Slots'].clip(lower=1))

# Merge into modelling_df
df = d2.merge(zone_exposure[['Zone_Int_ID', 'log_exposure']], 
                                   on='Zone_Int_ID', how='left')

# print(df['log_exposure'].describe())
df

,Zone_Int_ID,Time_Stamp,Mean_Severity,Mean_Distance(mi),Mean_Temp(F),Mean_Humidity(%),Mean_Pressure(in),Mean_Visibility(mi),Mean_Wind(mph),Mean_Precip(in),...,Weather_Clear,Weather_Cloudy,Weather_Dust/Windy,Weather_Rain/Drizzle,Weather_Snow/Ice,Weather_Stormy,Weather_Visibility,Day_Night,Accident_Count,log_exposure
0,0,2016-06-14 20:00:00,2.0,0.000,86.0,66.0,29.84,8.0,9.2,0.0,...,True,False,False,False,False,False,False,1,1,9.947696
1,0,2016-06-21 18:00:00,2.0,0.000,82.4,79.0,30.02,10.0,9.2,0.0,...,True,False,False,False,False,False,False,1,1,9.947696
2,0,2016-06-21 20:00:00,2.0,0.000,84.2,66.0,30.01,10.0,8.1,0.0,...,True,False,False,False,False,False,False,1,1,9.947696
3,0,2016-06-22 12:00:00,2.0,0.033,93.2,49.0,30.03,10.0,5.8,0.0,...,False,True,False,False,False,False,False,1,1,9.947696
4,0,2016-06-22 14:00:00,2.0,0.000,94.1,46.5,30.00,10.0,8.1,0.0,...,False,True,False,False,False,False,False,1,2,9.947696
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
351945,829,2022-09-16 18:00:00,2.0,0.506,81.0,85.0,30.03,10.0,0.0,0.0,...,False,True,False,False,False,False,False,0,1,2.484907
351946,830,2022-09-21 06:00:00,2.0,0.125,78.0,96.0,29.85,10.0,0.0,0.0,...,True,False,False,False,False,False,False,1,1,2.484907
351947,831,2022-12-07 16:00:00,2.0,0.063,72.0,84.0,30.19,10.0,6.0,0.0,...,True,False,False,False,False,False,False,0,1,2.484907
351948,832,2022-12-22 16:00:00,2.0,2.241,83.0,58.0,29.93,10.0,8.0,0.0,...,False,True,False,False,False,False,False,1,1,2.484907


In [6]:
feature_cols = [
    'Hour', 'Day_of_Week', 'Month', 'Weekend', 'Holiday',
    'City_Houston', 'City_LA', 'City_Miami',
    'Mean_Temp(F)', 'Mean_Humidity(%)', 'Mean_Pressure(in)',
    'Mean_Visibility(mi)', 'Mean_Wind(mph)', 'Mean_Precip(in)',
    'Amenity', 'Crossing', 'Give_Way', 'Junction', 'Railway',
    'Station', 'Stop', 'Traffic_Signal',
    'Weather_Clear', 'Weather_Cloudy', 'Weather_Rain/Drizzle',
    'Weather_Stormy', 'Weather_Visibility', 'Weather_Dust/Windy',
    'Day_Night'
]

# Convert all boolean columns to int
bool_cols = df[feature_cols].select_dtypes(include='bool').columns.tolist()
print(f"Converting: {bool_cols}")
df[bool_cols] = df[bool_cols].astype(int)

# Verify
print(df[feature_cols].dtypes)

Converting: ['Weekend', 'City_Houston', 'City_LA', 'City_Miami', 'Amenity', 'Crossing', 'Give_Way', 'Junction', 'Railway', 'Station', 'Stop', 'Traffic_Signal', 'Weather_Clear', 'Weather_Cloudy', 'Weather_Rain/Drizzle', 'Weather_Stormy', 'Weather_Visibility', 'Weather_Dust/Windy']
Hour                      int64
Day_of_Week               int64
Month                     int64
Weekend                   int64
Holiday                   int64
City_Houston              int64
City_LA                   int64
City_Miami                int64
Mean_Temp(F)            float64
Mean_Humidity(%)        float64
Mean_Pressure(in)       float64
Mean_Visibility(mi)     float64
Mean_Wind(mph)          float64
Mean_Precip(in)         float64
Amenity                   int64
Crossing                  int64
Give_Way                  int64
Junction                  int64
Railway                   int64
Station                   int64
Stop                      int64
Traffic_Signal            int64
Weather_Clear  

In [ ]:
df.head()

,Zone_Int_ID,Time_Stamp,Mean_Severity,Mean_Distance(mi),Mean_Temp(F),Mean_Humidity(%),Mean_Pressure(in),Mean_Visibility(mi),Mean_Wind(mph),Mean_Precip(in),...,Weather_Dust/Windy,Weather_Rain/Drizzle,Weather_Snow/Ice,Weather_Stormy,Weather_Visibility,Day_Night,Accident_Count,log_exposure_x,log_exposure_y,log_exposure
0,0,2016-06-14 20:00:00,2.0,0.000,86.0,66.0,29.84,8.0,9.2,0.0,...,0,0,False,0,0,1,1,9.947696,9.947696,9.947696
1,0,2016-06-21 18:00:00,2.0,0.000,82.4,79.0,30.02,10.0,9.2,0.0,...,0,0,False,0,0,1,1,9.947696,9.947696,9.947696
2,0,2016-06-21 20:00:00,2.0,0.000,84.2,66.0,30.01,10.0,8.1,0.0,...,0,0,False,0,0,1,1,9.947696,9.947696,9.947696
3,0,2016-06-22 12:00:00,2.0,0.033,93.2,49.0,30.03,10.0,5.8,0.0,...,0,0,False,0,0,1,1,9.947696,9.947696,9.947696
4,0,2016-06-22 14:00:00,2.0,0.000,94.1,46.5,30.00,10.0,8.1,0.0,...,0,0,False,0,0,1,2,9.947696,9.947696,9.947696


In [7]:
print(f"modelling_df shape: {df.shape}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

modelling_df shape: (351950, 37)
Memory usage: 101.7 MB


In [7]:
import statsmodels.api as sm
from sklearn.linear_model import TweedieRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np

mean = df['Accident_Count'].mean()
variance = df['Accident_Count'].var()
alpha = (variance - mean) / mean
print(f"Using alpha: {alpha:.3f}")

feature_cols = [
    'Hour', 'Day_of_Week', 'Month', 'Weekend', 'Holiday',
    'City_Houston', 'City_LA', 'City_Miami',
    'Mean_Temp(F)', 'Mean_Humidity(%)', 'Mean_Pressure(in)',
    'Mean_Visibility(mi)', 'Mean_Wind(mph)', 'Mean_Precip(in)',
    'Amenity', 'Crossing', 'Give_Way', 'Junction', 'Railway',
    'Station', 'Stop', 'Traffic_Signal',
    'Weather_Clear', 'Weather_Cloudy', 'Weather_Rain/Drizzle',
    'Weather_Stormy', 'Weather_Visibility', 'Weather_Dust/Windy',
    'Day_Night'
]

X = df[feature_cols].values
y = df['Accident_Count'].values
weights = df['log_exposure'].values

# Scale features for better convergence
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X_scaled, y, weights, test_size=0.2, random_state=42
)

# power=1.5 is between Poisson (1.0) and Gamma (2.0) — approximates Negative Binomial
model = TweedieRegressor(power=1.5, alpha=alpha, link='log', max_iter=1000)
model.fit(X_train, y_train, sample_weight=w_train)

# Evaluate
y_pred = model.predict(X_test)
print(f"Mean Absolute Error: {np.mean(np.abs(y_test - y_pred)):.3f}")
print(f"Mean predicted count: {y_pred.mean():.3f}")
print(f"Mean actual count:    {y_test.mean():.3f}")

# Feature importance (coefficients)
coef_df = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient': model.coef_
}).sort_values('Coefficient', ascending=False)
print(coef_df)

Using alpha: 1.129
Mean Absolute Error: 0.632
Mean predicted count: 1.415
Mean actual count:    1.432
                 Feature  Coefficient
19               Station     0.092252
7             City_Miami     0.058698
15              Crossing     0.051545
14               Amenity     0.035227
17              Junction     0.026157
20                  Stop     0.023504
22         Weather_Clear     0.022240
25        Weather_Stormy     0.020184
24  Weather_Rain/Drizzle     0.018224
23        Weather_Cloudy     0.018123
28             Day_Night     0.017169
21        Traffic_Signal     0.011462
12        Mean_Wind(mph)     0.009659
0                   Hour     0.007689
26    Weather_Visibility     0.005913
11   Mean_Visibility(mi)     0.004457
18               Railway     0.004072
8           Mean_Temp(F)     0.003621
27    Weather_Dust/Windy     0.000697
13       Mean_Precip(in)     0.000654
1            Day_of_Week    -0.000725
16              Give_Way    -0.001461
2                  Month